# 🌾 HarvestForecast — Day 37 | 100 Days of Projects

**Task:** Historical crop-yield prediction using country/region, crop, year and weather/agricultural covariates when verified present.  
**Dataset:** `HarvestForecast dataset` (five CSV files; original files never modified).  
**Scientific rule:** A year-based holdout, validation-only champion selection, once-only final test, and explicit warnings against causal/farm-level forecasting claims.

> **Critical interpretation:** These records are aggregated historical observations; even a strong score cannot validate individual-farm or future-climate predictions. Weather covariates measured over the target year may constitute *retrospective estimation* instead of an actionable pre-season forecast. See the leakage and deployment notes below.

**Notebook stages:** Environment → provenance/inventory → schema audit → target/feature eligibility → row and time audit → temporal split → preprocessing/baselines → model search → validation diagnostics → final held-out evaluation → artifacts and manifest.

Use Kaggle **CPU** (GPU not needed). Run All / Save Version after adding the dataset. Outputs go to `/kaggle/working/HarvestForecast_Day37/`.

In [1]:
# 01 — Environment & determinism (Kaggle CPU)
import os, sys, json, re, warnings, hashlib, platform, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
INPUT_ROOT = Path(os.environ.get('HARVEST_INPUT_ROOT', '/kaggle/input/datasets/wafabila/harvestforecast-dataset'))
if not INPUT_ROOT.exists():
    # Compatibility with alternate Kaggle dataset mount paths, never silently use an unrelated dataset.
    candidates = [Path('/kaggle/input/harvestforecast-dataset'), Path('/kaggle/input/datasets/wafabila/harvestforecast-dataset')]
    INPUT_ROOT = next((p for p in candidates if p.exists()), INPUT_ROOT)
OUT = Path(os.environ.get('HARVEST_OUTPUT_ROOT', '/kaggle/working/HarvestForecast_Day37'))
for folder in ('reports','assets/plots','assets/examples','models'):
    (OUT/folder).mkdir(parents=True, exist_ok=True)

CONFIG = dict(project='HarvestForecast', day=37, seed=SEED, input_root=str(INPUT_ROOT), output_root=str(OUT),
              target_selection='verified yield per area only', split='strict ascending calendar years',
              model_selection='validation MAE only', test_selection=False,
              intended_use='historical area/crop yield benchmarking; prospective availability unverified',
              input_availability='retrospective unless weather/pesticide fields known at prediction time')
print('Python:',sys.version.split()[0], 'pandas:',pd.__version__, 'scikit-learn:',sklearn.__version__)
print('Input:', INPUT_ROOT, '| exists:', INPUT_ROOT.exists())
print('Output:', OUT)
assert INPUT_ROOT.exists(), f'Attach HarvestForecast dataset first. Missing: {INPUT_ROOT}'

Python: 3.13.16 pandas: 2.3.3 scikit-learn: 1.6.1
Input: /kaggle/input/datasets/wafabila/harvestforecast-dataset | exists: True
Output: /kaggle/working/HarvestForecast_Day37


## 1 — Dataset inventory & provenance

Only inspect original files. The five sources may have overlapping records. **Do not concatenate** `yield.csv`, `df_yield_merged.csv`, rainfall, pesticides and temperature tables without verifying keys, units and join cardinality. The curated merged table is used as an experimental starting point, not treated as automatically trustworthy.

In [2]:
# 02 — Required file inventory, content hashes, source placeholders
REQUIRED = ['df_yield_merged.csv','yield.csv','rainfall.csv','temp.csv','pesticides.csv']
found = {}
for name in REQUIRED:
    candidates = [p for p in INPUT_ROOT.rglob(name) if p.is_file()]
    assert len(candidates)==1, f'Expected exactly one {name}, found {len(candidates)}: {candidates}'
    found[name]=candidates[0]

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

inventory=[]
for name,path in found.items():
    rec={'name':name, 'relative_path':str(path.relative_to(INPUT_ROOT)), 'bytes':path.stat().st_size,
         'sha256':sha256_file(path)}
    inventory.append(rec)
    print(f"{name:<24} {rec['bytes']/1024**2:>7.2f} MiB | SHA256 {rec['sha256'][:14]}…")
pd.DataFrame(inventory).to_csv(OUT/'reports/dataset_inventory.csv',index=False)
provenance={'local_kaggle_path':str(INPUT_ROOT),'accessed_utc':datetime.now(timezone.utc).isoformat(),
            'files':inventory,'claimed_upstream_reference':'https://zenodo.org/records/18761663',
            'upstream_url_verified_in_notebook':False,
            'license_verified':False,
            'note':'Upstream provenance reference is a lead from the collection notes, NOT proof of exact file equivalence or permission; verify official file hashes/version/license before public reuse.'}
(OUT/'reports/dataset_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')

df_yield_merged.csv         1.23 MiB | SHA256 0a2b4e025610b5…
yield.csv                   3.86 MiB | SHA256 ddd4855e82cc07…
rainfall.csv                0.13 MiB | SHA256 684df19bb1bdc2…
temp.csv                    1.36 MiB | SHA256 0635487775a894…
pesticides.csv              0.43 MiB | SHA256 913fa4d98285a1…


1444

In [3]:
# 03 — Read all five tables for audit (never train on raw auxiliary tables directly)
tables={}
for name,path in found.items():
    try:
        df=pd.read_csv(path,low_memory=False)
    except UnicodeDecodeError:
        df=pd.read_csv(path,encoding='latin-1',low_memory=False)
    tables[name]=df
    print(f'\n{name}: shape={df.shape}, columns={list(df.columns)}')
    display(df.head(2))

schema=[]
for name,df in tables.items():
    for col in df.columns:
        schema.append({'source':name,'column':str(col),'dtype':str(df[col].dtype),
                       'missing':int(df[col].isna().sum()),'non_null':int(df[col].notna().sum()),
                       'unique':int(df[col].nunique(dropna=True))})
pd.DataFrame(schema).to_csv(OUT/'reports/schema_audit.csv',index=False)


df_yield_merged.csv: shape=(25932, 7), columns=['area', 'year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp', 'item', 'value_hg_ha']


,area,year,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp,item,value_hg_ha
0,Albania,1990,1485.0,121.0,16.37,Maize,36613
1,Albania,1990,1485.0,121.0,16.37,Potatoes,66667



yield.csv: shape=(56717, 12), columns=['Domain Code', 'Domain', 'Area Code', 'Area', 'Element Code', 'Element', 'Item Code', 'Item', 'Year Code', 'Year', 'Unit', 'Value']


,Domain Code,Domain,Area Code,Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value
0,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1961,1961,hg/ha,14000
1,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1962,1962,hg/ha,14000



rainfall.csv: shape=(6727, 3), columns=[' Area', 'Year', 'average_rain_fall_mm_per_year']


,Area,Year,average_rain_fall_mm_per_year
0,Afghanistan,1985,327
1,Afghanistan,1986,327



temp.csv: shape=(71311, 3), columns=['year', 'country', 'avg_temp']


,year,country,avg_temp
0,1849,Côte D'Ivoire,25.58
1,1850,Côte D'Ivoire,25.52



pesticides.csv: shape=(4349, 7), columns=['Domain', 'Area', 'Element', 'Item', 'Year', 'Unit', 'Value']


,Domain,Area,Element,Item,Year,Unit,Value
0,Pesticides Use,Albania,Use,Pesticides (total),1990,tonnes of active ingredients,121.0
1,Pesticides Use,Albania,Use,Pesticides (total),1991,tonnes of active ingredients,121.0


## 2 — Target identification and leakage checks

The target **must** represent agricultural yield *per area* (e.g. hg/ha), not total production tonnage. The notebook deliberately stops if no eligible target is identified or multiple candidate columns are ambiguous. Features derived from yield/production are excluded. Crop, area, year and potentially measured rainfall/temperature/pesticides are candidate explanatory variables, not guaranteed pre-planting forecasts.

In [4]:
# 04 — Column discovery with transparent mapping
raw=tables['df_yield_merged.csv'].copy()

def norm(s): return re.sub(r'[^a-z0-9]+','_',str(s).strip().lower()).strip('_')
lookup={norm(c):c for c in raw.columns}
print('Normalized columns:',lookup)

def resolve_exact(candidates, required=False):
    matches=[lookup[c] for c in candidates if c in lookup]
    matches=list(dict.fromkeys(matches))
    if required and len(matches)!=1:
        raise ValueError(f'Cannot unambiguously resolve {candidates}. Found {matches}. Review raw columns.')
    return matches[0] if len(matches)==1 else None

TARGET=resolve_exact(['value_hg_ha','hg_ha_yield','yield_hg_ha','yield_hg_per_ha','yield_kg_ha','yield_kg_per_ha','yield_t_ha','yield_t_per_ha'],required=True)
YEAR=resolve_exact(['year','harvest_year'],required=True)
AREA=resolve_exact(['area','country','region'],required=True)
CROP=resolve_exact(['item','crop','crop_type'],required=True)
print('Selected:',{'target':TARGET,'year':YEAR,'area':AREA,'crop':CROP})
TARGET_UNIT = ('hg/ha' if 'hg' in norm(TARGET) else 'kg/ha' if 'kg' in norm(TARGET) else 't/ha')
CONFIG.update(target=TARGET,target_unit=TARGET_UNIT,year=YEAR,area=AREA,crop=CROP)

# Exclude all other outcomes, derived outputs, identifiers and potentially predictive target proxies.
forbidden_tokens=['yield','production','harvest_output','output_tonnes','output_tons','target','index','unnamed','id']
blocked=[]
for col in raw.columns:
    c=norm(col)
    if col in (TARGET,YEAR,AREA,CROP): continue
    if any(c == tok or c.startswith(tok + '_') or c.endswith('_' + tok) for tok in forbidden_tokens): blocked.append(col)
# If an auxiliary table repeats the target, no auxiliary tables are joined; reduce leak surface.
CONFIG['blocked_columns']=blocked
print('Blocked potentially leaky columns:',blocked)

Normalized columns: {'area': 'area', 'year': 'year', 'average_rain_fall_mm_per_year': 'average_rain_fall_mm_per_year', 'pesticides_tonnes': 'pesticides_tonnes', 'avg_temp': 'avg_temp', 'item': 'item', 'value_hg_ha': 'value_hg_ha'}
Selected: {'target': 'value_hg_ha', 'year': 'year', 'area': 'area', 'crop': 'item'}
Blocked potentially leaky columns: []


In [5]:
# 05 — Missing/duplicate/target quality audit
source_rows=len(raw)
raw['__year_num']=pd.to_numeric(raw[YEAR],errors='coerce')
raw['__target_num']=pd.to_numeric(raw[TARGET],errors='coerce')
raw['__area_norm']=raw[AREA].astype('string').str.strip().str.lower()
raw['__crop_norm']=raw[CROP].astype('string').str.strip().str.lower()

invalid=(raw['__year_num'].isna() | raw['__target_num'].isna() | (raw['__target_num']<=0) |
         raw['__area_norm'].isna() | raw['__crop_norm'].isna() |
         raw['__area_norm'].eq('') | raw['__crop_norm'].eq(''))
invalid_n=int(invalid.sum())
clean=raw.loc[~invalid].copy()
clean[YEAR]=clean['__year_num'].astype(int)
clean[TARGET]=clean['__target_num'].astype(float)
clean[AREA]=clean[AREA].astype(str).str.strip()
clean[CROP]=clean[CROP].astype(str).str.strip()
clean=clean.drop(columns=['__year_num','__target_num','__area_norm','__crop_norm'])
exact_dup=int(clean.duplicated().sum())
clean=clean.drop_duplicates().copy()

# Identical Area/Item/Year rows may have conflicting targets; exclude all of those keys.
keys=[AREA,CROP,YEAR]
group=clean.groupby(keys,dropna=False)[TARGET].agg(['size','nunique'])
conflicting_keys=group[group['nunique']>1]
repeated_keys=group[group['size']>1]
if len(repeated_keys):
    print('REVIEW: Repeated area/crop/year keys:',len(repeated_keys),'conflicting target keys:',len(conflicting_keys))
if len(conflicting_keys):
    bad_keys=set(conflicting_keys.index.tolist())
    clean=clean.loc[~clean[keys].apply(tuple,axis=1).isin(bad_keys)].copy()
    print('Quarantined all rows with conflicting keys.')
# Leave one observation per key, avoiding inflated evidence due to repeated area/year records.
key_dup=int(clean.duplicated(keys).sum())
clean=clean.drop_duplicates(keys,keep='first').copy()
clean=clean.sort_values([YEAR,AREA,CROP]).reset_index(drop=True)
assert len(clean)>=80, 'Too few valid observations for temporal ML.'
assert clean[YEAR].nunique()>=7, 'At least 7 distinct years required to make meaningful year holdouts.'

quality={'source_rows':int(source_rows),'invalid_rows_quarantined':invalid_n,'exact_duplicates_removed':exact_dup,
         'repeated_area_crop_year_keys':int(len(repeated_keys)),
         'conflicting_target_keys_quarantined':int(len(conflicting_keys)),
         'duplicate_area_crop_year_rows_removed_after_conflict_check':key_dup,
         'retained_rows':int(len(clean)), 'year_min':int(clean[YEAR].min()), 'year_max':int(clean[YEAR].max()),
         'years':int(clean[YEAR].nunique()), 'countries_or_areas':int(clean[AREA].nunique()),
         'crops':int(clean[CROP].nunique()),'target_unit':TARGET_UNIT}
print(json.dumps(quality,indent=2))
(OUT/'reports/dataset_audit.json').write_text(json.dumps(quality,indent=2),encoding='utf-8')

REVIEW: Repeated area/crop/year keys: 3690 conflicting target keys: 0
{
  "source_rows": 25932,
  "invalid_rows_quarantined": 0,
  "exact_duplicates_removed": 0,
  "repeated_area_crop_year_keys": 3690,
  "conflicting_target_keys_quarantined": 0,
  "duplicate_area_crop_year_rows_removed_after_conflict_check": 12802,
  "retained_rows": 13130,
  "year_min": 1990,
  "year_max": 2013,
  "years": 23,
  "countries_or_areas": 101,
  "crops": 10,
  "target_unit": "hg/ha"
}


398

In [6]:
# 06 — Inspect missingness, broad ranges, candidate covariate availability
safe_extra=[]
for col in clean.columns:
    if col in [AREA,CROP,YEAR,TARGET] or col in blocked: continue
    if norm(col) in ('unnamed_0','index'): continue
    if pd.api.types.is_numeric_dtype(clean[col]):
        safe_extra.append(col)
    else:
        # Reject arbitrary object columns, often identifiers and text leakage.
        print('[EXCLUDED nonnumeric extra]',col)

# Validate time-independent non-target covariates, preserving only plausible weather/agricultural measurements.
allowed_context_tokens=('rain','temp','pesticide','humidity','precipitation','radiation','soil','wind')
excluded_extra=[c for c in safe_extra if not any(t in norm(c) for t in allowed_context_tokens)]
extra=[c for c in safe_extra if c not in excluded_extra]
print('Candidate weather/agriculture covariates:',extra)
print('Excluded ambiguous numeric extras:',excluded_extra)
missing_report=clean[[AREA,CROP,YEAR,TARGET]+extra].isna().mean().sort_values(ascending=False).rename('missing_fraction')
display(missing_report.to_frame())
missing_report.to_csv(OUT/'reports/feature_missingness.csv')
CONFIG['features']=[AREA,CROP,YEAR]+extra
CONFIG['feature_temporal_warning']='Weather, annual precipitation, pesticides may only be known after harvest; this benchmark is retrospective, not pre-season operational.'
assert len(extra)>0, 'No verified contextual numeric covariates available. Review merged schema before continuing.'

Candidate weather/agriculture covariates: ['average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']
Excluded ambiguous numeric extras: []


,missing_fraction
area,0.0
item,0.0
year,0.0
value_hg_ha,0.0
average_rain_fall_mm_per_year,0.0
pesticides_tonnes,0.0
avg_temp,0.0


In [7]:
# 07 — Core EDA with truthful plots
fig,axs=plt.subplots(1,3,figsize=(17,4.4))
axs[0].hist(clean[TARGET],bins=45,alpha=.85)
axs[0].set(title=f'Yield distribution ({TARGET_UNIT})',xlabel=TARGET_UNIT,ylabel='Rows')
annual=clean.groupby(YEAR)[TARGET].median()
axs[1].plot(annual.index,annual.values,marker='o',markersize=2)
axs[1].set(title='Historical median yield by year',xlabel='Year',ylabel=TARGET_UNIT)
clean[CROP].value_counts().head(12).sort_values().plot.barh(ax=axs[2])
axs[2].set(title='Most frequent crops',xlabel='Rows')
fig.tight_layout(); fig.savefig(OUT/'assets/plots/dataset_eda.png',dpi=140,bbox_inches='tight');plt.show();plt.close(fig)

## 3 — Chronological train / validation / untouched test

**No random row split:** contiguous years are assigned to train, then validation, then test. This evaluates generalization to later historical years, while the observation-level region/crop overlaps may persist. The notebook counts overlap and does not claim independent-region generalization. Train-time features using actual weather/rainfall of test years are **retrospective**, not deployable forecasting features unless separate ex-ante predictions are supplied.

In [8]:
# 08 — Hold out complete final years. Selection uses validation ONLY.
years=sorted(clean[YEAR].unique().tolist())
ny=len(years)
n_test=max(2,int(np.ceil(ny*.15)))
n_val=max(2,int(np.ceil(ny*.15)))
assert ny-n_test-n_val>=3, f'Not enough training years: {ny}'
train_years=years[:ny-n_val-n_test]; val_years=years[ny-n_val-n_test:ny-n_test]; test_years=years[ny-n_test:]
train=clean[clean[YEAR].isin(train_years)].copy()
val=clean[clean[YEAR].isin(val_years)].copy()
test=clean[clean[YEAR].isin(test_years)].copy()
assert min(val_years)>max(train_years) and min(test_years)>max(val_years)
assert min(len(train),len(val),len(test))>0

def pairs(df):return set(zip(df[AREA].str.lower().str.strip(),df[CROP].str.lower().str.strip()))
ptr,pva,pte=pairs(train),pairs(val),pairs(test)
split_info={'train':{'rows':len(train),'year_start':min(train_years),'year_end':max(train_years)},
            'validation':{'rows':len(val),'year_start':min(val_years),'year_end':max(val_years)},
            'test':{'rows':len(test),'year_start':min(test_years),'year_end':max(test_years)},
            'area_crop_pairs_train_val_overlap':len(ptr&pva),
            'area_crop_pairs_train_test_overlap':len(ptr&pte),
            'note':'No row can share a year across splits. Area/crop identities can recur over years; geographic OOD validation not claimed.'}
print(json.dumps(split_info,indent=2))
(OUT/'reports/split_audit.json').write_text(json.dumps(split_info,indent=2),encoding='utf-8')
# Write only row signatures/indices; raw dataset is not repackaged into repository.
for title,frame in [('train',train),('validation',val),('test',test)]:
    membership=frame[[AREA,CROP,YEAR]].copy()
    membership['sample_key_sha256']=membership.astype(str).agg('|'.join,axis=1).map(lambda s:hashlib.sha256(s.encode()).hexdigest())
    membership[['sample_key_sha256',YEAR]].to_csv(OUT/f'reports/{title}_split_manifest.csv',index=False)

{
  "train": {
    "rows": 8444,
    "year_start": 1990,
    "year_end": 2005
  },
  "validation": {
    "rows": 2331,
    "year_start": 2006,
    "year_end": 2009
  },
  "test": {
    "rows": 2355,
    "year_start": 2010,
    "year_end": 2013
  },
  "area_crop_pairs_train_val_overlap": 581,
  "area_crop_pairs_train_test_overlap": 580,
  "note": "No row can share a year across splits. Area/crop identities can recur over years; geographic OOD validation not claimed."
}


In [9]:
# 09 — Feature preprocessing and leakage guard
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

FEATURES=CONFIG['features']
assert TARGET not in FEATURES and len(set(FEATURES))==len(FEATURES)
for banned in blocked: assert banned not in FEATURES
Xtr=train[FEATURES]; Xv=val[FEATURES]
ytr=train[TARGET].values.astype('float64'); yv=val[TARGET].values.astype('float64')

def make_encoder():
    try:return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError:return OneHotEncoder(handle_unknown='ignore',sparse=False)

def preprocess():
    return ColumnTransformer([
        ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',make_encoder())]),[AREA,CROP]),
        ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),[YEAR]+extra),
    ],remainder='drop',sparse_threshold=0)

def metrics(ytrue, ypred):
    return {'mae':float(mean_absolute_error(ytrue,ypred)),
            'rmse':float(np.sqrt(mean_squared_error(ytrue,ypred))),
            'r2':float(r2_score(ytrue,ypred)) if len(ytrue)>=2 else None,
            'wape_percent':float(100*np.sum(np.abs(ytrue-ypred))/np.sum(np.abs(ytrue))) if np.sum(np.abs(ytrue))>0 else None}

def build(est):return Pipeline([('preprocess',preprocess()),('regressor',est)])

## 4 — Baselines & candidate regressors

Train on **training years only** and rank on **validation MAE** (lower is better). Include a global median dummy and a crop–area historical median baseline to avoid presenting a sophisticated model as useful when trivial seasonal persistence could be better. Comparisons are observational, not causal.

In [10]:
# 10 — Historical area-crop median baseline (no validation/test target fit)
group_median=train.groupby([AREA,CROP])[TARGET].median().to_dict()
area_median=train.groupby(AREA)[TARGET].median().to_dict()
crop_median=train.groupby(CROP)[TARGET].median().to_dict()
global_median=float(train[TARGET].median())
def historic_predict(df):
    return np.asarray([group_median.get((r[AREA],r[CROP]),
                      crop_median.get(r[CROP],area_median.get(r[AREA],global_median)))
                      for _,r in df[[AREA,CROP]].iterrows()],dtype='float64')

history_val=historic_predict(val)
records=[{'model':'historical_area_crop_median','stage':'validation',**metrics(yv,history_val)}]
print('Historical median baseline:',records[-1])

Historical median baseline: {'model': 'historical_area_crop_median', 'stage': 'validation', 'mae': 15518.21063921064, 'rmse': 31219.37622027403, 'r2': 0.8606050748810785, 'wape_percent': 20.26607813141308}


In [11]:
# 11 — Training, resource-aware candidates. No test data read here.
models={
    'dummy_median':DummyRegressor(strategy='median'),
    'ridge_alpha_1':Ridge(alpha=1.0),
    'ridge_alpha_100':Ridge(alpha=100.0),
    'random_forest_120':RandomForestRegressor(n_estimators=120,min_samples_leaf=3,max_features=.8,
                                            n_jobs=-1,random_state=SEED),
    'random_forest_200':RandomForestRegressor(n_estimators=200,min_samples_leaf=6,max_features=.8,
                                            n_jobs=-1,random_state=SEED),
    'hist_gb':HistGradientBoostingRegressor(max_iter=180,max_leaf_nodes=20,l2_regularization=1,
                                           random_state=SEED),
}
fitted={}; predictions_val={'historical_area_crop_median':history_val}
for name, estimator in models.items():
    start=time.time()
    model=build(estimator)
    model.fit(Xtr,ytr)
    pred=model.predict(Xv)
    assert np.isfinite(pred).all(), f'Nonfinite predictions for {name}'
    fitted[name]=model; predictions_val[name]=pred
    score=metrics(yv,pred)
    records.append({'model':name,'stage':'validation','seconds':round(time.time()-start,2),**score})
    print(f'{name:<25} MAE={score["mae"]:,.2f} RMSE={score["rmse"]:,.2f} R2={score["r2"]:.4f}')

leader=pd.DataFrame(records).sort_values(['mae','rmse','model']).reset_index(drop=True)
leader['rank']=np.arange(1,len(leader)+1)
leader.to_csv(OUT/'reports/validation_leaderboard.csv',index=False)
display(leader)

dummy_median              MAE=56,412.69 RMSE=92,733.58 R2=-0.2299
ridge_alpha_1             MAE=29,866.23 RMSE=44,409.27 R2=0.7179
ridge_alpha_100           MAE=31,028.57 RMSE=49,696.81 R2=0.6468
random_forest_120         MAE=11,232.15 RMSE=21,881.98 R2=0.9315
random_forest_200         MAE=12,967.59 RMSE=25,325.53 R2=0.9083
hist_gb                   MAE=16,282.15 RMSE=27,184.86 R2=0.8943


,model,stage,mae,rmse,r2,wape_percent,seconds,rank
0,random_forest_120,validation,11232.150701,21881.982055,0.931519,14.668679,1.97,1
1,random_forest_200,validation,12967.586740,25325.529125,0.908269,16.935079,2.75,2
2,historical_area_crop_median,validation,15518.210639,31219.376220,0.860605,20.266078,NaN,3
3,hist_gb,validation,16282.148884,27184.863292,0.894305,21.263747,1.07,4
4,ridge_alpha_1,validation,29866.226572,44409.271261,0.717937,39.003935,0.08,5
5,ridge_alpha_100,validation,31028.574374,49696.809426,0.646772,40.521909,0.07,6
6,dummy_median,validation,56412.691334,92733.581452,-0.229908,73.672412,0.05,7


In [12]:
# 12 — Choose solely by validation data. Lock selection before test.
CHAMPION=str(leader.iloc[0]['model'])
champion_val=predictions_val[CHAMPION]
selection={'champion':CHAMPION,'selection_metric':'validation_mae','lower_is_better':True,
           'validation_mae':float(leader.iloc[0]['mae']),
           'test_used_to_select_model':False,'selection_timestamp_utc':datetime.now(timezone.utc).isoformat()}
(OUT/'reports/model_selection.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
print('CHAMPION LOCKED:',CHAMPION)
print('Selection only from validation MAE:',selection['validation_mae'])

CHAMPION LOCKED: random_forest_120
Selection only from validation MAE: 11232.150701067401


In [13]:
# 13 — Validation error diagnostics by crop and unseen pair
val_diag=val[[AREA,CROP,YEAR,TARGET]].copy()
val_diag['prediction']=champion_val
val_diag['absolute_error']=(val_diag[TARGET]-val_diag['prediction']).abs()
val_diag['known_area_crop']=[(a,c) in ptr for a,c in zip(val_diag[AREA],val_diag[CROP])]
by_crop=val_diag.groupby(CROP).agg(n=('absolute_error','size'),mae=('absolute_error','mean')).reset_index().sort_values('mae',ascending=False)
by_crop.to_csv(OUT/'reports/validation_mae_by_crop.csv',index=False)
val_diag.nlargest(30,'absolute_error').to_csv(OUT/'assets/examples/validation_error_cases.csv',index=False)
fig,ax=plt.subplots(figsize=(6,5))
ax.scatter(yv,champion_val,s=9,alpha=.32)
xy=[min(yv.min(),champion_val.min()),max(yv.max(),champion_val.max())]
ax.plot(xy,xy,'--',linewidth=1)
ax.set(xlabel=f'Observed ({TARGET_UNIT})',ylabel=f'Predicted ({TARGET_UNIT})',title='Validation: observed vs predicted')
fig.tight_layout();fig.savefig(OUT/'assets/plots/validation_scatter.png',dpi=140);plt.show();plt.close(fig)
display(by_crop.head(15))

,item,n,mae
3,Potatoes,376,22685.001666
2,Plantains and others,80,22551.709213
7,Sweet potatoes,204,20817.749913
0,Cassava,164,20064.751445
9,Yams,80,13210.570010
1,Maize,360,7302.762528
4,"Rice, paddy",262,5953.459280
8,Wheat,323,4223.064299
5,Sorghum,254,4018.112564
6,Soybeans,228,2985.565349


## 5 — FINAL HELD-OUT TEST (once only)

**Freeze champion before this cell.** The final test covers future years relative to train/validation and must not be used to change model, feature list, thresholds, preprocessing or split. The exact chosen model is evaluated without refitting on validation, to keep the train-only model-selection evidence transparent.

In [14]:
# 14 — Single final test evaluation
Xte=test[FEATURES]; yte=test[TARGET].to_numpy(dtype='float64')
if CHAMPION=='historical_area_crop_median':
    pred_test=historic_predict(test)
else:
    pred_test=fitted[CHAMPION].predict(Xte)
assert np.isfinite(pred_test).all()
result=metrics(yte,pred_test)
result.update({'champion':CHAMPION,'test_rows':len(test),'test_years':[min(test_years),max(test_years)],
              'unit':TARGET_UNIT,'test_used_for_champion_selection':False})
print('FINAL HELD-OUT TEST:',json.dumps(result,indent=2))
(OUT/'reports/heldout_test_metrics.json').write_text(json.dumps(result,indent=2),encoding='utf-8')

test_diag=test[[AREA,CROP,YEAR,TARGET]].copy()
test_diag['prediction']=pred_test
test_diag['absolute_error']=np.abs(test_diag[TARGET]-test_diag['prediction'])
test_diag['known_area_crop']=[(a,c) in ptr for a,c in zip(test_diag[AREA],test_diag[CROP])]
test_diag.nlargest(40,'absolute_error').to_csv(OUT/'assets/examples/heldout_test_failure_cases.csv',index=False)
by_crop_test=test_diag.groupby(CROP).agg(n=('absolute_error','size'),mae=('absolute_error','mean')).reset_index().sort_values('mae',ascending=False)
by_crop_test.to_csv(OUT/'reports/heldout_test_mae_by_crop.csv',index=False)
by_year_test=test_diag.groupby(YEAR).agg(n=('absolute_error','size'),mae=('absolute_error','mean')).reset_index()
by_year_test.to_csv(OUT/'reports/heldout_test_mae_by_year.csv',index=False)
display(by_crop_test.head(12))

FINAL HELD-OUT TEST: {
  "mae": 16393.50783840602,
  "rmse": 30993.078389200196,
  "r2": 0.8742591993483355,
  "wape_percent": 20.156111055216666,
  "champion": "random_forest_120",
  "test_rows": 2355,
  "test_years": [
    2010,
    2013
  ],
  "unit": "hg/ha",
  "test_used_for_champion_selection": false
}


,item,n,mae
7,Sweet potatoes,206,33793.665411
0,Cassava,164,32048.086325
3,Potatoes,378,31696.912216
2,Plantains and others,84,24999.165589
9,Yams,82,21982.224626
1,Maize,366,10920.179318
4,"Rice, paddy",266,9459.072912
5,Sorghum,260,5811.483725
8,Wheat,322,5342.542536
6,Soybeans,227,3353.484982


In [15]:
# 15 — Test plots (reporting only; never tune)
fig,axs=plt.subplots(1,2,figsize=(12,4.6))
axs[0].scatter(yte,pred_test,s=9,alpha=.3)
lo=min(np.min(yte),np.min(pred_test)); hi=max(np.max(yte),np.max(pred_test))
axs[0].plot([lo,hi],[lo,hi],'--',lw=1)
axs[0].set(title='Held-out test: observed vs predicted',xlabel=f'Observed ({TARGET_UNIT})',ylabel=f'Predicted ({TARGET_UNIT})')
axs[1].hist(yte-pred_test,bins=45)
axs[1].set(title='Held-out residuals',xlabel=f'Actual − predicted ({TARGET_UNIT})',ylabel='Frequency')
fig.tight_layout();fig.savefig(OUT/'assets/plots/heldout_test_diagnostics.png',dpi=140,bbox_inches='tight');plt.show();plt.close(fig)

## 6 — Export & model card

**Deployment contract:** Model outputs predict the target in its original unit (`hg/ha`, `kg/ha` or `t/ha`). This notebook does **not** convert it to kilograms of harvest for a given farm; production estimates would require accurate field area and crop management context. The reported model is a historical panel-data benchmark, not a validated farm agronomy advisor.

In [16]:
# 16 — Persist champion and preprocessing
# Baseline uses a dict of historical medians; sklearn models include their fitted preprocess pipeline.
if CHAMPION=='historical_area_crop_median':
    artifact={'kind':'historical_area_crop_median','area':AREA,'crop':CROP,'target':TARGET,
              'group_median':group_median,'crop_median':crop_median,'area_median':area_median,
              'global_median':global_median, 'features':FEATURES,'unit':TARGET_UNIT}
else:
    artifact={'kind':'sklearn_pipeline','model':fitted[CHAMPION],'features':FEATURES,
              'target':TARGET,'unit':TARGET_UNIT}
model_path=OUT/'models/harvestforecast_champion.joblib'
joblib.dump(artifact,model_path,compress=3)

# Validate roundtrip without touching test evaluation decisions.
loaded=joblib.load(model_path)
if loaded['kind']=='sklearn_pipeline':
    check=loaded['model'].predict(Xv.head(min(len(Xv),5)))
else:
    check=historic_predict(val.head(min(len(val),5)))
assert np.isfinite(check).all(), 'Artifact roundtrip invalid'
CONFIG.update(champion=CHAMPION,feature_columns=FEATURES,target=TARGET,target_unit=TARGET_UNIT,
              input_data_path=str(found['df_yield_merged.csv']))
(OUT/'reports/run_config.json').write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False),encoding='utf-8')

model_card=f"""# HarvestForecast — Day 37 Model Card

## Task
Retrospective country/area × crop × year agricultural yield regression.

## Dataset
`df_yield_merged.csv` inside HarvestForecast Kaggle input; other four CSVs audited only and not automatically rejoined.

## Outcome
`{TARGET}` in **{TARGET_UNIT}**. Champion `{CHAMPION}` selected exclusively using validation **MAE**.

## Split
Train {min(train_years)}–{max(train_years)} ({len(train):,} records); validation {min(val_years)}–{max(val_years)} ({len(val):,}); held-out test {min(test_years)}–{max(test_years)} ({len(test):,}).

## Held-out metrics (not used for tuning)
MAE **{result['mae']:,.3f} {TARGET_UNIT}**; RMSE **{result['rmse']:,.3f} {TARGET_UNIT}**; R² **{result['r2']:.4f}**.

## Limitations
- Weather and pesticide variables for a target year may be **post-hoc**, not available at planting time.
- Country-level/area-level aggregates cannot be assumed valid for individual fields.
- Area/crop identities may overlap across years; generalization to unseen locations is not measured.
- Records may inherit unverified original collection methods, licenses and measurement noise.
- Climate shifts, nonstationarity and data availability can degrade future-year performance.
- Do not treat yield predictions as guarantees or automatically recommend resource input levels.
- Historical median baseline is not a trained neural network and has a different inference implementation.

## Reproduction
Seed `{SEED}`. `run_config.json`, `dataset_inventory.csv`, `dataset_audit.json`, `split_audit.json`, leaderboard, per-crop and per-year evaluation and plot artifacts are emitted next to this card.
"""
(OUT/'reports/MODEL_CARD.md').write_text(model_card,encoding='utf-8')
print('Saved:',model_path, '(',round(model_path.stat().st_size/1024**2,2),'MiB)')

Saved: /kaggle/working/HarvestForecast_Day37/models/harvestforecast_champion.joblib ( 7.18 MiB)


In [17]:
# 17 — Output inventory, manifest and ZIP (excluding original Kaggle data)
import zipfile
paths=sorted(p for p in OUT.rglob('*') if p.is_file() and p.name!='artifact_manifest.json')
manifest=[]
for p in paths:
    manifest.append({'path':p.relative_to(OUT).as_posix(),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
(OUT/'artifact_manifest.json').write_text(json.dumps({'project':'HarvestForecast','day':37,'files':manifest},indent=2),encoding='utf-8')
package=OUT.parent/'HarvestForecast_Day37_RESULTS.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(OUT.parent).as_posix())
with zipfile.ZipFile(package) as z:
    bad=z.testzip(); assert bad is None, f'ZIP CRC failed: {bad}'
print('RESULTS:',OUT)
print('ZIP:',package)
print('ZIP size MiB:',round(package.stat().st_size/1024**2,2))
print('Files:',len(manifest)+1)
print('FINAL MAE:',result['mae'],TARGET_UNIT)

RESULTS: /kaggle/working/HarvestForecast_Day37
ZIP: /kaggle/working/HarvestForecast_Day37_RESULTS.zip
ZIP size MiB: 7.87
Files: 24
FINAL MAE: 16393.50783840602 hg/ha


## Final checklist

- **Verify** source dataset license, version and source checksums against its official publisher before public distribution.
- **Inspect** `reports/dataset_audit.json`, `split_audit.json`, `validation_leaderboard.csv` and `heldout_test_metrics.json`.
- **Report** true held-out metrics only after successful Save Version. Never invent accuracy, robustness, calibrated confidence, or production forecasts.
- **For pre-season forecast claims:** replace realized target-year rainfall, temperature and pesticides with genuinely available *at-prediction-time* weather forecasts/historical normals and rerun the evaluation from the beginning.
- **GitHub:** commit the notebook, metadata, reports and small plots, not the Kaggle raw dataset or `.joblib` model by default.

**Result bundle:** `/kaggle/working/HarvestForecast_Day37_RESULTS.zip`